Please run this script on Colab or any other virtual environment in order to protect your computer as this script runs a heavy task.

In [ ]:
# 1. Install dependencies in Colab
!pip install transformers pandas torch

import pandas as pd
import torch
from transformers import pipeline
from tqdm import tqdm

# 2. Check for GPU (Crucial for speed)
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

# 3. Load FinBERT onto the GPU
finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert", device=device)

# 4. Load your Kaggle/ResearchGate dataset
# Ensure your dataframe has 'Date' and 'Headline' columns
df = pd.read_csv('YOUR_RAW_NEWS_DATASET.csv', on_bad_lines='skip', engine='python')

# Target 'publish_date' and convert YYYYMMDD integers to datetime
df['Date'] = pd.to_datetime(df['date']).dt.date

# 5. The Batch Processing Engine
daily_stress = []
grouped = df.groupby('Date')

for date, group in tqdm(grouped, desc="Processing Days"):
    headlines = group['news'].tolist()

    # Process all headlines for the day in one GPU batch
    try:
        results = finbert(headlines, batch_size=64, truncation=True, max_length=128)

        # Calculate how much "Panic" exists today
        negative_scores = [res['score'] for res in results if res['label'] == 'negative']
        daily_score = sum(negative_scores) / len(negative_scores) if negative_scores else 0.0

        daily_stress.append({'Date': date, 'Stress_Score': daily_score})
    except Exception as e:
        print(f"Error on {date}: {e}")

# 6. Export the clean historical matrix
stress_df = pd.DataFrame(daily_stress)
stress_df.to_csv('data/historical_nlp_stress_rg.csv', index=False)